### Cafe's TI repo kaggle implemented by Xynon (heavily based on **andite's** colab implementation with mods).

Textual Inversion repo from **Cafe**. Thanks for this wonderful TI repo, all credit goes to him.

Much of the code was borrowed so big thanks to **andite** for the original TI google colab, we all love you bro :)

<img src="https://media.tenor.com/kCZjTqCKiggAAAAC/hug.gif" width="350">
    
### Discord:

> Cafe - `Starport — かふぇ#0438`

> andite - `andite#8484`
    
> Xynon - `Xynon#7407`
    
(I don't do technical support guarantees, also please don't contact andite for this)

### If you find any errors in this notebook, feel free to contact me on discord.

*Disclaimer: this notebook is harder to use (and probably buggier) than the google colab version. If you are looking for convenience or have no idea how this works just use the colab (you can find that here: https://colab.research.google.com/drive/1bbtGmH0XfQWzVKROhiIP8x5EAv6XuohJ?usp=sharing).*

*It has better instructions too... that actually exist.*

> **Updates [12/27/22]**

*   lazy config blind fix (oh no)

> **Updates [12/25/22]**

*   Ayy it's a christmas day update!
*   Did some more minor cleanup
*   Added logs deleter
*   Added lazy mode config editor (courtesy of legekka#4242's lazy edition colab), untested for now
*   Switched to latest environment, may break things
*   Happy holidays everyone!

<img src=https://media.tenor.com/FHoJOz1mxW0AAAAd/illo-navidad.gif width="400">

> **Updates [12/24/22]**


*   Cleaned up, removed some stuff required in colab but not kaggle
*   Synced up to andite colab torch version fix

> **Updates [11/19/22]**


*   Added title markdown for more explanation about the notebook.
*   Added easy log downloader so this notebook should be essentially code free to use.

<img src=https://media.tenor.com/Fi5H8EfqtFAAAAAC/yay-yeah.gif width="200">

> **Updates [11/18/22]**


*   Moved things over to /kaggle/tmp as kaggle persistence is horrifyingly slow (better just to redownload model & dataset from hf).
*   Initial release.

# **Check the GPU.** 

turn on accelerators from 3 dot menu on top right bar (this notebook was developed using a P100). 

*Note: 2xT4 may net you more total VRAM but require some modifications (the --gpu number way down below).*

In [ ]:
!nvidia-smi

# **huggingface_hub** login.

Get the token from https://huggingface.co/settings/tokens

*Note: the ui will not refresh after inputting the token and clicking login (this works in colab but not here for some reason).*

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

# **Grab your model**. 

Input HF repo and filepath in the code block below.

*Note: Use a huggingface model type repo for the model you'll be using in training.*

In [ ]:
path = "experimentals/experimental12b.ckpt"
repo = "Xynon/models"

from huggingface_hub import hf_hub_download
hf_hub_download(repo_id=repo, filename=path, cache_dir="/kaggle/tmp")

# **In case your model folders need cleansing.** 
*Note: this will wipe your repo models, do not run if this is not what you want.*

In [ ]:
repolist = repo.split("/")
hfpath = repolist[1]+"--"+repolist[0]+"--"+repolist[1]

import shutil
shutil.rmtree('/kaggle/tmp/'+hfpath)

# **Grab your dataset**. 

Input HF repo and filepath into the code block below.

*Note: Make sure to use a huggingface dataset type repo as well, not a model one.*

In [ ]:
datapath = "exampledataset.zip"

pathlist = datapath.split("/")
namex = pathlist[-1]
name = namex.split(".")[0]

datarepo = "Xynon/datasets"

datarepolist = datarepo.split("/")
hfdatapath = datarepolist[1]+"--"+datarepolist[0]+"--"+datarepolist[1]

from huggingface_hub import hf_hub_download
hf_hub_download(repo_id=datarepo, repo_type="dataset", filename=namex, cache_dir="/kaggle/tmp")
!mkdir -p "/kaggle/tmp/images-for-training/{name}"
!7z x "/kaggle/tmp/{hfdatapath}/snapshots/*/{namex}" -o"/kaggle/tmp/images-for-training/{name}" -y

# **In case your dataset folders need cleansing.** 
*Note: this will wipe your repo dataset folder, do not run if this is not what you want.*

In [ ]:
import shutil
shutil.rmtree('/kaggle/tmp/'+hfdatapath)
shutil.rmtree('/kaggle/tmp/images-for-training/*')

# **Download the textual inversion repo.**
*Note: No need to run this if you already have the repo (with persistence), but if it's your first time using this notebook, run the cell.*

In [ ]:
%cd /kaggle/working/
!mkdir /kaggle/working/modelsbackup/
!mkdir /kaggle/working/stable-textual-inversion-cafe/
!mkdir /kaggle/working/stable-textual-inversion-cafe/logs/
!mv /kaggle/working/stable-textual-inversion-cafe/logs/ /kaggle/working/modelsbackup/
!rm -rf /kaggle/working/stable-textual-inversion-cafe/
!git clone https://github.com/Raearn/stable-textual-inversion-cafe.git
%cd /kaggle/working/stable-textual-inversion-cafe
print("Done.")

# **Install python packages.**

*Note: It is highly recommended to install the packages or else the training won't work.*

In [ ]:
%cd /kaggle/working/stable-textual-inversion-cafe
!pip install omegaconf einops pytorch-lightning==1.6.5 test-tube transformers kornia -e git+https://github.com/CompVis/taming-transformers.git@master#egg=taming-transformers -e git+https://github.com/openai/CLIP.git@main#egg=clip
!pip install setuptools==59.5.0
!pip install pillow==9.0.1
!pip install torchmetrics==0.6.0
!pip install torch==1.12.1+cu113 torchvision==0.13.1+cu113 torchaudio==0.12.1 --extra-index-url https://download.pytorch.org/whl/cu113
!pip install torchtext==0.13.1
!pip install -e .
print("Done.")

# Edit the yaml file first based from the config you will choose in the cell before training!

> artstyle - `/kaggle/working/stable-textual-inversion-cafe/configs/stable-diffusion/artstyle.yaml`

> character - `/kaggle/working/stable-textual-inversion-cafe/configs/stable-diffusion/character.yaml`


In line 29, set your vector tokens depending on the amount of training images you have.

>`45+ images - 8 tokens`

>`100-150+ images - 16 tokens`

>`300++ images - 20 or more tokens`

Optional: In line 27, you can add some additional initializer words inside the square brackets if you want the training to focus more or make a certain part more accurate. (I am not talking about the initializer_word inside the train cell.)

>For example, put `"hair"` inside the square brackets if you want it to focus more on the accuracy for the hair especially if it's complex or unique. You could also put words like `"painting"` or `"lighting"`.

Optional: You can set a custom learning rate in line 2, but `5.0e-03` / `0.005` is the default and the safest value.

Training preview images and checkpoints/pt files should be in `/kaggle/working/stable-textual-inversion-cafe/logs`

> Note: The actual training preview images are named `samples_scaled_gs` from the logs, which means, `inputs_gs`, `reconstruction_gs`, `samples_gs`,` aren't the actual **training preview images**. Ignore them.

### **Lazy Method (courtesy of legekka#4242):**
Untested for now, use at your own risk.

In [ ]:
#Change this stuff
project_name = "example"                    
initializer_words = ["girl","face","hair","eyes","illustratrion","clothes"]
num_vectors_per_token = 12                                          
ti_type = "character"
model = "/kaggle/tmp/models--Xynon--models/snapshots/*/experimentals/experimental12b.ckpt"

#Don't change this stuff
# loading the base config
import yaml
with open("/kaggle/working/stable-textual-inversion-cafe/configs/stable-diffusion/" + ti_type + ".yaml", "r") as f:
    base_config = yaml.safe_load(f)

# calulate the number of vectors per token from the number of images
if num_vectors_per_token == "auto":
    num_vectors_per_token = round(num_images / 8.5)
    if num_vectors_per_token < 8:
        num_vectors_per_token = 8
    if num_vectors_per_token > 18:
        num_vectors_per_token = 18

# calculating max steps
if num_images <= 110:
    max_steps = 11000
else:
    max_steps = 16000
import math
repeats = math.ceil(max_steps / num_images)

# setting the parameters
base_config["model"]["params"]["personalization_config"]["params"]["initializer_words"] = initializer_words
base_config["model"]["params"]["personalization_config"]["params"]["num_vectors_per_token"] = num_vectors_per_token
base_config["data"]["params"]["train"]["params"]["repeats"] = repeats
base_config["lightning"]["trainer"]["max_steps"] = max_steps
base_config["lightning"]["modelcheckpoint"]["params"]["every_n_train_steps"] = 500
base_config["lightning"]["callbacks"]["image_logger"]["params"]["batch_frequency"] = 2000
base_config["model"]["params"]["log_every_t"] = 500

# saving the config (your_example_project_name_here.yaml)
with open("/kaggle/working/stable-textual-inversion-cafe/configs/stable-diffusion/" + project_name + ".yaml", "w") as f:
    yaml.dump(base_config, f)

### **Full Method:** 
Run the code block below (with %load) then modify, uncomment and replace %load with %%writefile to write to the file.

In [ ]:
%load "/kaggle/working/stable-textual-inversion-cafe/configs/stable-diffusion/artstyle.yaml"

# **Train.**

In [ ]:
#Change the stuff below
model = "/kaggle/tmp/models--Xynon--models/snapshots/*/experimentals/experimental12b.ckpt"
dataset = "/kaggle/tmp/images-for-training/exampledataset/exampledataset/"
project_name = "example"
initializer_word = "illustration"

#Don't change this stuff
artstyle = "configs/stable-diffusion/artstyle.yaml"
character = "configs/stable-diffusion/character.yaml"
lazy = "configs/stable-diffusion/" + project_name + ".yaml"

#Change the stuff below
config = artstyle #Either artstyle, character or lazy 

#Don't change this stuff
%cd /kaggle/working/stable-textual-inversion-cafe
!python "main.py" \
 --base {config} \
 -t --no-test \
 --actual_resume {model} \
 -n {project_name} \
 --gpus 1 \
 --data_root {dataset} \
 --init_word {initializer_word}

# **Resume training here.**

*Note: Install python packages first before retraining or else it won't work!*

*Note: Don't forget to edit your config file, should be the same setting as the one you did for the TI you trained.*

In [ ]:
#Change the stuff below
model = "/kaggle/tmp/models--Xynon--models/snapshots/*/experimentals/experimental12b.ckpt"
last_embed = "/kaggle/working/stable-textual-inversion-cafe/logs/example2022-11-12T12-52-32_example/checkpoints/embeddings_gs-500.pt"
last_checkpoint = "/kaggle/working/stable-textual-inversion-cafe/logs/example2022-11-12T12-52-32_example/checkpoints/last.ckpt"
dataset = "/kaggle/tmp/images-for-training/exampledataset/exampledataset/"
project_name = "example"
initializer_word = "illustration"

#Don't change this stuff
artstyle = "configs/stable-diffusion/artstyle.yaml"
character = "configs/stable-diffusion/character.yaml"
lazy = "configs/stable-diffusion/" + project_name + ".yaml"

#Change the stuff below
config = artstyle #Either artstyle, character, or lazy

#Don't change this stuff
%cd /kaggle/working/stable-textual-inversion-cafe
!python "main.py" \
 --base {config} \
 -t --no-test \
 --actual_resume {model} \
 --gpus 1 \
 --data_root {dataset} \
 --embedding_manager_ckpt {last_embed} \
 --resume_from_checkpoint {last_checkpoint} \
 -n {project_name} \
 --init_word {initializer_word}

# **Download embeds here.**
*Note: this will download the entire logs folder (embeds, samples, and stuff).*

In [ ]:
%cd /kaggle/working/stable-textual-inversion-cafe/logs/
!7z a /kaggle/working/logs.zip ./ -y
import os
from IPython.display import FileLink 
%cd /kaggle/working/
FileLink(r'logs.zip')

# **In case your logs folders need cleansing.** 
*Note: this will wipe your logs (output) folder, do not run if this is not what you want.*

In [ ]:
import shutil
shutil.rmtree('/kaggle/working/stable-textual-inversion-cafe/logs/')